# CorpBrain — Notebook 4: Local RAG Demo

**100% local — no API keys, no internet (after install).**

**What this demonstrates:**
- Store meeting transcripts in ChromaDB with sentence-transformer embeddings
- Semantic search: find most relevant chunks for a question
- Extractive Q&A: answer questions from retrieved chunks
- Duplicate task detection using vector similarity

> This is exactly what `knowledge_service` does in the full pipeline.

## Step 1 — Install

In [1]:
!pip install chromadb sentence-transformers -q
print("Libraries ready")


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Libraries ready


## Step 2 — Setup ChromaDB + Embedding Model

In [2]:
import chromadb
from sentence_transformers import SentenceTransformer
import uuid

print("Loading sentence-transformer (all-MiniLM-L6-v2)...")
embedder   = SentenceTransformer('all-MiniLM-L6-v2')
chroma     = chromadb.Client()  # in-memory for demo; use PersistentClient in production
collection = chroma.get_or_create_collection('meeting_transcripts')
print(f"ChromaDB ready — collection: 'meeting_transcripts'")

Loading sentence-transformer (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ChromaDB ready — collection: 'meeting_transcripts'


## Step 3 — Store 3 Sample Meeting Transcripts

In [3]:
meetings = [
    {
        "id": "mtg-001",
        "transcript": "Sprint 14 planning meeting. Ahmed will complete the React Native setup by Wednesday. Sara needs to fix the authentication bug — blocking all users. We decided to deploy to staging on Monday. Main blocker: CI pipeline is taking 45 minutes. Khaled will optimize the CI this week.",
        "date": "2026-04-28",
    },
    {
        "id": "mtg-002",
        "transcript": "Sprint 13 retrospective. What went well: we delivered all committed stories and the API is faster. What to improve: code reviews are too slow, sometimes 3 days. Action items: Omar will set up a 24-hour review SLA. Sara will add unit tests for the payment module by Friday.",
        "date": "2026-04-21",
    },
    {
        "id": "mtg-003",
        "transcript": "Emergency standup. The payment service is down since 2am. Ahmed rolled back the deployment. Root cause: database migration script had a bug. Sara will write a fix and push by noon. Khaled will add a rollback test to the CI pipeline. No other blockers.",
        "date": "2026-04-25",
    },
]

def chunk_text(text, size=50, overlap=10):
    words  = text.split()
    chunks = []
    for i in range(0, len(words), size - overlap):
        chunks.append(' '.join(words[i:i+size]))
    return chunks

total_chunks = 0
for meeting in meetings:
    chunks = chunk_text(meeting['transcript'])
    for i, chunk in enumerate(chunks):
        emb = embedder.encode(chunk).tolist()
        collection.add(
            ids=[f"{meeting['id']}_chunk_{i}"],
            embeddings=[emb],
            documents=[chunk],
            metadatas=[{"meeting_id": meeting['id'], "date": meeting['date']}],
        )
        total_chunks += 1

print(f"Stored {len(meetings)} meetings → {total_chunks} chunks in ChromaDB")

Stored 3 meetings → 6 chunks in ChromaDB


## Step 4 — Semantic Search

In [4]:
questions = [
    "Who is fixing the CI pipeline?",
    "What is the deadline for the payment bug fix?",
    "What went well in the retrospective?",
    "Who is responsible for code review policy?",
    "What caused the payment service outage?",
]

print("Semantic Search Results:")
print("=" * 70)
for question in questions:
    q_emb   = embedder.encode(question).tolist()
    results = collection.query(query_embeddings=[q_emb], n_results=1)
    
    best_chunk = results['documents'][0][0]
    meeting_id = results['metadatas'][0][0]['meeting_id']
    distance   = results['distances'][0][0]
    
    print(f"Q: {question}")
    print(f"   Best match [{meeting_id}] (distance={distance:.3f}):")
    print(f"   → {best_chunk[:100]}...")
    print()

Semantic Search Results:
Q: Who is fixing the CI pipeline?
   Best match [mtg-003] (distance=0.867):
   → pipeline. No other blockers....

Q: What is the deadline for the payment bug fix?
   Best match [mtg-002] (distance=1.055):
   → unit tests for the payment module by Friday....

Q: What went well in the retrospective?
   Best match [mtg-002] (distance=1.153):
   → Sprint 13 retrospective. What went well: we delivered all committed stories and the API is faster. W...

Q: Who is responsible for code review policy?
   Best match [mtg-002] (distance=1.534):
   → Sprint 13 retrospective. What went well: we delivered all committed stories and the API is faster. W...

Q: What caused the payment service outage?
   Best match [mtg-003] (distance=0.676):
   → Emergency standup. The payment service is down since 2am. Ahmed rolled back the deployment. Root cau...



## Step 5 — Extractive Q&A

In [5]:
def extractive_answer(question: str, top_k: int = 2) -> dict:
    '''Find the most relevant sentence to the question using word overlap.'''
    q_emb   = embedder.encode(question).tolist()
    results = collection.query(query_embeddings=[q_emb], n_results=top_k)
    
    q_words = set(question.lower().split())
    best_sentence, best_score, best_meeting = '', 0, 'unknown'
    
    for docs, metas in zip(results['documents'], results['metadatas']):
        for chunk, meta in zip(docs, metas):
            for sentence in chunk.split('. '):
                overlap = len(q_words & set(sentence.lower().split()))
                if overlap > best_score:
                    best_score    = overlap
                    best_sentence = sentence.strip()
                    best_meeting  = meta['meeting_id']
    
    return {
        'answer': f"[{best_meeting}]: {best_sentence}" if best_sentence else 'Not found',
        'sources': len(results['documents'][0]),
    }

print("Extractive Q&A Results:")
print("=" * 70)
for question in questions:
    result = extractive_answer(question)
    print(f"Q: {question}")
    print(f"A: {result['answer']}")
    print()

Extractive Q&A Results:
Q: Who is fixing the CI pipeline?
A: [mtg-003]: The payment service is down since 2am

Q: What is the deadline for the payment bug fix?
A: [mtg-002]: unit tests for the payment module by Friday.

Q: What went well in the retrospective?
A: [mtg-002]: What went well: we delivered all committed stories and the API is faster

Q: Who is responsible for code review policy?
A: [mtg-002]: What went well: we delivered all committed stories and the API is faster

Q: What caused the payment service outage?
A: [mtg-003]: The payment service is down since 2am



## Step 6 — Duplicate Task Detection

In [6]:
def check_duplicate(task: str, threshold: float = 0.20) -> dict:
    emb     = embedder.encode(task).tolist()
    results = collection.query(query_embeddings=[emb], n_results=1)
    distance = results['distances'][0][0]
    
    if distance < threshold:
        return {
            'is_duplicate': True,
            'distance': round(distance, 4),
            'similar_to': results['documents'][0][0][:80],
            'meeting_id': results['metadatas'][0][0]['meeting_id'],
        }
    return {'is_duplicate': False, 'distance': round(distance, 4)}

new_tasks = [
    "Optimize the CI pipeline to reduce build time",      # duplicate of mtg-001
    "Add 24-hour SLA for code reviews",                   # duplicate of mtg-002
    "Build a new mobile app from scratch using Flutter",  # NOT a duplicate
    "Fix authentication bug in login flow",               # duplicate of mtg-001
]

print("Duplicate Detection:")
print(f"{'Duplicate?':12} | {'Distance':8} | Task")
print("-" * 70)
for task in new_tasks:
    result = check_duplicate(task)
    icon = "⚠️ DUP" if result['is_duplicate'] else "✅ NEW"
    print(f"{icon:12} | {result['distance']:.4f}   | {task[:50]}")
    if result['is_duplicate']:
        print(f"             Similar to: {result['similar_to'][:60]}")
print()
print("✅ RAG demo complete — all local, zero API keys!")

Duplicate Detection:
Duplicate?   | Distance | Task
----------------------------------------------------------------------
✅ NEW        | 1.0608   | Optimize the CI pipeline to reduce build time
✅ NEW        | 0.9146   | Add 24-hour SLA for code reviews
✅ NEW        | 1.5457   | Build a new mobile app from scratch using Flutter
✅ NEW        | 1.4756   | Fix authentication bug in login flow

✅ RAG demo complete — all local, zero API keys!
